# Adversarial Attacks and Defenses on Cloud-Based Medical Diagnostics

This Colab notebook reproduces the full project workflow from a fresh runtime:

1. Clone the repository and install dependencies.
2. Load the chest X-ray dataset from Google Drive.
3. Train the baseline model.
4. Evaluate FGSM adversarial attacks.
5. Evaluate defensive randomization.
6. Train an adversarially trained model.
7. Compare baseline vs defended model.
8. Run LLM advisory through local Ollama.
9. Generate the final results dashboard.

Before running, set Colab to GPU:

`Runtime -> Change runtime type -> T4 GPU`

Expected runtime: about 45-75 minutes.


## Cell 1: Mount Google Drive, Clone Repo, Checkout Dashboard Branch, Install Dependencies

This project expects the dataset zip to be available in Google Drive as:

`/content/drive/MyDrive/raw.zip`


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = "https://github.com/syedmaaiz/-G4_Adversarial-Attacks-and-Defenses-on-Cloud-Based-Medical-Diagnostics-with-LLM-Advisory.git"
REPO_DIR = "/content/-G4_Adversarial-Attacks-and-Defenses-on-Cloud-Based-Medical-Diagnostics-with-LLM-Advisory"

import os

if not os.path.exists(REPO_DIR):
    !git clone --branch dashboard-ui-updates {REPO_URL}
else:
    %cd {REPO_DIR}
    !git fetch
    !git checkout dashboard-ui-updates
    !git pull origin dashboard-ui-updates

%cd {REPO_DIR}
!pip install -r requirements.txt


## Cell 2: Unzip Dataset and Verify Layout

This handles both possible zip layouts:

- `data/raw/train/...`
- `data/raw/raw/train/...`


In [ ]:
import os

%cd /content/-G4_Adversarial-Attacks-and-Defenses-on-Cloud-Based-Medical-Diagnostics-with-LLM-Advisory

DATA_ZIP = "/content/drive/MyDrive/raw.zip"
assert os.path.exists(DATA_ZIP), f"Dataset zip not found at {DATA_ZIP}"

!rm -rf data/raw
!mkdir -p data/raw
!unzip -q "{DATA_ZIP}" -d data/raw

if os.path.exists("data/raw/raw/train"):
    !mv data/raw/raw/* data/raw/
    !rmdir data/raw/raw || true

print("Dataset folders:")
!find data/raw -maxdepth 3 -type d | sort

assert os.path.exists("data/raw/train/NORMAL")
assert os.path.exists("data/raw/train/PNEUMONIA")
assert os.path.exists("data/raw/test/NORMAL")
assert os.path.exists("data/raw/test/PNEUMONIA")
print("Dataset layout is correct.")


## Cell 3: Train Baseline Model

This trains the class-weighted baseline ResNet18 model and saves:

`artifacts/models/baseline_model.pt`

Expected clean test accuracy is around `0.85-0.86`.


In [ ]:
%cd /content/-G4_Adversarial-Attacks-and-Defenses-on-Cloud-Based-Medical-Diagnostics-with-LLM-Advisory

!rm -f artifacts/models/baseline_model.pt
!python -m src.models.train_baseline --epochs 8 --batch-size 32 --pretrained


## Cell 4: Evaluate FGSM Attack on Baseline

This shows how much the attack hurts the original model.

Expected approximate result:

- Clean accuracy: `0.8574`
- Adversarial accuracy: `0.5369`
- Standard attack success rate: `0.3738`


In [ ]:
%cd /content/-G4_Adversarial-Attacks-and-Defenses-on-Cloud-Based-Medical-Diagnostics-with-LLM-Advisory

!python -m src.attacks.evaluate_fgsm --checkpoint artifacts/models/baseline_model.pt --epsilon 0.01


## Cell 5: Evaluate Simple Defense: Defensive Randomization

This defense should help a little, but not enough to be the final recommended defense.

Expected approximate result:

- Adversarial accuracy: `0.5369`
- Defended adversarial accuracy: `0.5705`
- Defense recovery rate: `0.1050`


In [ ]:
%cd /content/-G4_Adversarial-Attacks-and-Defenses-on-Cloud-Based-Medical-Diagnostics-with-LLM-Advisory

!python -m src.defenses.evaluate_randomization --epsilon 0.01 --resize-delta 8


## Cell 6: Train Stronger Defense: Adversarial Training

This trains a second model using both clean images and FGSM-attacked images. It saves:

`artifacts/models/adversarial_model.pt`

Expected approximate result:

- Clean test accuracy: around `0.8590`
- Test adversarial accuracy at epsilon `0.01`: around `0.7404`


In [ ]:
%cd /content/-G4_Adversarial-Attacks-and-Defenses-on-Cloud-Based-Medical-Diagnostics-with-LLM-Advisory

!rm -f artifacts/models/adversarial_model.pt
!python -m src.defenses.train_adversarial --epochs 8 --batch-size 32 --pretrained --epsilon 0.01


## Cell 7: Compare Baseline vs Adversarially Trained Model

This is the key defense result.

Expected comparison:

- Baseline adversarial accuracy: about `0.5369`
- Adversarially trained adversarial accuracy: about `0.7404`
- Baseline attack success rate: about `0.3738`
- Adversarially trained attack success rate: about `0.1381`


In [ ]:
%cd /content/-G4_Adversarial-Attacks-and-Defenses-on-Cloud-Based-Medical-Diagnostics-with-LLM-Advisory

print("BASELINE MODEL AGAINST FGSM")
!python -m src.attacks.evaluate_fgsm --checkpoint artifacts/models/baseline_model.pt --epsilon 0.01

print("\nADVERSARIALLY TRAINED MODEL AGAINST FGSM")
!python -m src.attacks.evaluate_fgsm --checkpoint artifacts/models/adversarial_model.pt --epsilon 0.01


## Cell 8: Install Ollama and Pull Local LLM

This avoids OpenAI/Gemini quota issues by running a local LLM inside Colab.


In [ ]:
%cd /content/-G4_Adversarial-Attacks-and-Defenses-on-Cloud-Based-Medical-Diagnostics-with-LLM-Advisory

!sudo apt-get update -qq
!sudo apt-get install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &

import time, os

time.sleep(5)
!curl -s http://127.0.0.1:11434/api/tags

!ollama pull qwen2.5:1.5b

os.environ["LLM_PROVIDER"] = "ollama"
os.environ["OLLAMA_MODEL"] = "qwen2.5:1.5b"
os.environ["OLLAMA_BASE_URL"] = "http://127.0.0.1:11434"
os.environ["LLM_TIMEOUT_SECONDS"] = "240"


## Cell 9: Run Pre-Defense and Post-Defense LLM Advisory

Pre-defense mode only sees the baseline attack result and recommends defenses to try.

Post-defense mode sees defense results and recommends which defense should be selected.


In [ ]:
%cd /content/-G4_Adversarial-Attacks-and-Defenses-on-Cloud-Based-Medical-Diagnostics-with-LLM-Advisory

print("PRE-DEFENSE LLM ADVICE")
!python -m src.llm_advisor.recommend --mode pre-defense

print("\nPOST-DEFENSE LLM ADVICE")
!python -m src.llm_advisor.recommend --mode post-defense


## Cell 10: Launch the Interactive Dashboard and Security Chatbot

This launches the responsive dashboard with experiment-stage buttons and the grounded Security Results Assistant. The chatbot uses the Ollama configuration from Cell 8 and falls back to built-in project answers if the LLM is temporarily unavailable.


In [ ]:
%cd /content/-G4_Adversarial-Attacks-and-Defenses-on-Cloud-Based-Medical-Diagnostics-with-LLM-Advisory

!python -m src.dashboard.generate_dashboard

import html
import subprocess
import time
from google.colab import output
from IPython.display import HTML, display

existing_server = globals().get("dashboard_server")
if existing_server is not None and existing_server.poll() is None:
    existing_server.terminate()
    existing_server.wait(timeout=10)

dashboard_log = open("dashboard_server.log", "w", encoding="utf-8")
dashboard_server = subprocess.Popen(
    ["uvicorn", "src.dashboard.chat_api:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=dashboard_log,
    stderr=subprocess.STDOUT,
)
time.sleep(3)
assert dashboard_server.poll() is None, open("dashboard_server.log", encoding="utf-8").read()

dashboard_url = output.eval_js("google.colab.kernel.proxyPort(8000)")
safe_dashboard_url = html.escape(dashboard_url, quote=True)
display(HTML(
    f'<a href="{safe_dashboard_url}" target="_blank" rel="noopener noreferrer" '
    'style="display:inline-block;padding:12px 18px;background:#2563eb;color:white;'
    'text-decoration:none;border-radius:8px;font-weight:600">'
    'Open Dashboard in a New Tab</a>'
))


## Optional Cell 11: Save Checkpoints to Google Drive

Use this if you want to preserve trained models after the Colab runtime disconnects.


In [ ]:
%cd /content/-G4_Adversarial-Attacks-and-Defenses-on-Cloud-Based-Medical-Diagnostics-with-LLM-Advisory

!mkdir -p "/content/drive/MyDrive/adversarial_project_checkpoints"
!cp artifacts/models/baseline_model.pt "/content/drive/MyDrive/adversarial_project_checkpoints/"
!cp artifacts/models/adversarial_model.pt "/content/drive/MyDrive/adversarial_project_checkpoints/"
!ls -lh "/content/drive/MyDrive/adversarial_project_checkpoints"


## Notes for Grader

- The dataset and trained model checkpoints are intentionally not committed to GitHub.
- The project is designed to train in Colab from `raw.zip`.
- Do not place API keys in this notebook. Ollama is used for the LLM advisory step to avoid cloud API quota issues.
- Slight differences in final numbers are acceptable because GPU training can vary.
